# 06 - Kaggle classifier v0.2 candidate

Research-only notebook for training and evaluating the experimental `mll23-leukemiaattri-dinobloom-b-mlp-v0.2` classifier head candidate. This notebook is not clinically validated and must not be interpreted as diagnostic software.

Restartability rule: every major stage writes a report or artifact under `OUTPUT_ROOT`; rerunning the notebook reuses completed artifacts unless inputs/configuration change.


## 1. Purpose and safety constraints

Goals:
- Preserve protected `mll23-dinobloom-b-mlp-v0.1` behavior.
- Train experimental 18-output MLP heads only; keep DinoBloom-B frozen.
- Evaluate whether LeukemiaAttri-compatible crops improve external generalization without MLL23 regression.
- Produce truthful reports even when acceptance fails.

Safety constraints:
- Do not overwrite or repackage v0.1.
- Do not create a production tag automatically.
- Do not claim improvement until all acceptance gates pass.
- Treat unmapped LeukemiaAttri classes as OOD/uncertainty evidence, not as correct MLL23 labels.


## 2. Configuration

Edit only this cell for Kaggle paths and experiment knobs. The notebook auto-discovers common artifact names under `/kaggle/input` and the cloned repo, so most paths can stay as `EDIT_ME` if the attached Kaggle dataset contains files/folders with the expected names. DinoBloom is downloaded during the run. LeukemiaAttri can be attached as a Kaggle input or downloaded with the guarded options below.


In [ ]:
from pathlib import Path

CONFIG = {
    "seed": 42,
    # If you git clone the repo in Kaggle, this is usually already correct.
    "repo_root": Path("/kaggle/working/blood"),
    "repo_git_url": "",  # optional: set if you want the notebook to clone the repo when missing
    # Downloaded during the run. Set dinobloom_download_url to the official/pinned source URL.
    "dinobloom_weights": Path("/kaggle/working/blood/models/backbones/dinobloom-b.pth"),
    "dinobloom_download_url": "EDIT_ME_OFFICIAL_DINOBLOOM_B_URL",
    "dinobloom_expected_sha256": "b1ea16884d1ac1331f9e2d80f730b1b72cab58c9d65cc8cc2ac7ec785a8213c7",
    # These can be left as EDIT_ME if you attach a Kaggle input containing files/folders with these names.
    # The environment cell auto-discovers them under /kaggle/input and common repo output paths.
    "mll23_embeddings": Path("/kaggle/input/EDIT_ME/mll23_embeddings.pt"),
    "mll23_manifest": Path("/kaggle/input/EDIT_ME/mll23_manifest.csv"),
    "mll23_splits": Path("/kaggle/input/EDIT_ME/mll23_splits.csv"),
    "classifier_v01_bundle": Path("/kaggle/input/EDIT_ME/mll23-dinobloom-b-mlp-v0.1"),
    "leukemiaattri_root": Path("/kaggle/input/EDIT_ME/LeukemiaAttri_Dataset"),
    # Optional LeukemiaAttri download path, matching notebook 05 behavior.
    "run_leukemia_download": False,
    "leukemia_zip_urls": [],  # Google Takeout zip URL(s), one or more; preferred for large Drive folders
    "leukemia_zip_files": [],  # already-attached /kaggle/input/*.zip file(s), optional
    "leukemia_gdrive_folder_url": "https://drive.google.com/drive/folders/1J5ld-tK6cewj9wXWUi3rs6UdlHnDBe8U?usp=sharing",
    "leukemia_download_output": Path("/kaggle/working/blood/data/raw/LeukemiaAttri"),
    "output_root": Path("/kaggle/working/classifier_v0_2"),
    "padding_ratios": [0.10, 0.25, 0.40, 0.60],
    "selected_padding_fallback": 0.25,
    "batch_size": 128,
    "epochs": 30,
    "patience": 6,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "gradient_clip_norm": 5.0,
    "lambda_mll23": 1.0,
    "lambda_leukemia_exact": 0.7,
    "lambda_leukemia_partial": 0.5,
    "run_all_four_candidates": True,
    "allow_outlier_exposure": False,
    "acceptance_gates": {
        "mll23_accuracy_max_drop": 0.005,
        "mll23_macro_f1_max_drop": 0.010,
        "external_exact_macro_f1_min_improvement": 0.05,
        "high_confidence_error_must_not_increase": True,
        "incorrect_accepted_must_not_increase": True,
        "calibration_materially_worse_allowed": False,
        "all_18_output_classes_required": True,
    },
}


## 3. Environment and reproducibility


In [ ]:
import csv
import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = CONFIG["repo_root"]
if not (REPO_ROOT / "src" / "bloodfilm").exists():
    if CONFIG.get("repo_git_url"):
        REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", str(CONFIG["repo_git_url"]), str(REPO_ROOT)], check=True)
    elif Path.cwd().joinpath("src", "bloodfilm").exists():
        REPO_ROOT = Path.cwd()
    else:
        raise RuntimeError(
            f"Repo root does not contain src/bloodfilm: {REPO_ROOT}. "
            "Clone the repo to /kaggle/working/blood or set CONFIG['repo_git_url']."
        )
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

print("Working directory:", REPO_ROOT)
print("Kaggle input roots:")
for item in sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []:
    print(" -", item)

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception:
    plt = None
    sns = None

from bloodfilm.classification.bundle import inspect_bundle, load_bundle_documents
from bloodfilm.classification.dinobloom import PREPROCESSING, extract_backbone_embedding, load_dinobloom_b_backbone, preprocess_crop
from bloodfilm.classification.heads import build_head
from bloodfilm.classification.inference import classify_probabilities
from bloodfilm.classification.uncertainty import UncertaintyPolicy, decide_prediction
from bloodfilm.classification.v02 import (
    MLL23_CLASSES,
    acceptance_status,
    assert_hashes_unchanged,
    decide_source_mapping,
    group_aware_split,
    hash_files,
    partial_label_loss,
    probability_schema,
    split_leakage_report,
    square_crop_geometry,
    stable_sample_hash,
    verify_mll23_class_order,
)
from bloodfilm.documents import load_config_document, write_json
from bloodfilm.evaluation.classifier import (
    calibration_summary,
    classification_report,
    confusion_matrix,
    detailed_per_class_metrics,
    expected_calibration_error,
    fit_temperature,
    reliability_bin_rows,
    top_k_accuracy,
)
from bloodfilm.util import sha256_file

SEED = int(CONFIG["seed"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as exc:
    print("Deterministic algorithm setting unavailable:", exc)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

OUTPUT_ROOT = CONFIG["output_root"]
REPORT_DIR = OUTPUT_ROOT / "outputs" / "reports"
PLOT_DIR = OUTPUT_ROOT / "outputs" / "plots"
CHECKPOINT_DIR = OUTPUT_ROOT / "outputs" / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "data" / "manifests"
EMBEDDING_DIR = OUTPUT_ROOT / "data" / "embeddings"
CROP_DIR = OUTPUT_ROOT / "data" / "crops"
MODEL_DIR = OUTPUT_ROOT / "models" / "mll23-leukemiaattri-dinobloom-b-mlp-v0.2"
for directory in [REPORT_DIR, PLOT_DIR, CHECKPOINT_DIR, MANIFEST_DIR, EMBEDDING_DIR, CROP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    return value

def write_report(path, payload):
    payload = {"schema_version": 1, **payload}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path

def is_placeholder(path):
    return "EDIT_ME" in str(path)

def input_candidates(pattern):
    roots = [Path("/kaggle/input"), REPO_ROOT]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(sorted(root.rglob(pattern)))
    return [path for path in matches if path.exists()]

def resolve_path(config_key, patterns, *, must_be_dir=False):
    configured = Path(CONFIG[config_key])
    if configured.exists() and not is_placeholder(configured):
        return configured
    for pattern in patterns:
        for candidate in input_candidates(pattern):
            if must_be_dir and not candidate.is_dir():
                continue
            if not must_be_dir and not candidate.is_file():
                continue
            print(f"Auto-discovered {config_key}: {candidate}")
            CONFIG[config_key] = candidate
            return candidate
    return configured

def download_file(url, output_path):
    if not url or str(url).startswith("EDIT_ME"):
        raise FileNotFoundError("DinoBloom weights are configured for download, but dinobloom_download_url is not set. Enable internet and set CONFIG['dinobloom_download_url'] to the official pinned URL.")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_path.with_suffix(output_path.suffix + ".part")
    print(f"Downloading DinoBloom-B weights to {output_path} ...")
    urllib.request.urlretrieve(str(url), tmp_path)
    tmp_path.replace(output_path)

def ensure_dinobloom_weights():
    weights_path = Path(CONFIG["dinobloom_weights"])
    expected_sha = str(CONFIG.get("dinobloom_expected_sha256") or "")
    if not weights_path.exists():
        download_file(CONFIG.get("dinobloom_download_url"), weights_path)
    actual_sha = sha256_file(weights_path)
    if expected_sha and actual_sha != expected_sha:
        raise RuntimeError(f"Downloaded DinoBloom-B SHA-256 mismatch: expected {expected_sha}, got {actual_sha}")
    return weights_path, actual_sha

def maybe_download_leukemiaattri():
    existing = resolve_path(
        "leukemiaattri_root",
        ["LeukemiaAttri_Dataset", "LeukemiaAttri", "leukemia-attri", "blood-cancer-dataset-leukemia-attri-miccai-2024"],
        must_be_dir=True,
    )
    if existing.exists() and not is_placeholder(existing):
        return existing
    if not CONFIG.get("run_leukemia_download"):
        return existing
    zip_urls = [str(item) for item in CONFIG.get("leukemia_zip_urls", []) if str(item).strip()]
    zip_files = [str(item) for item in CONFIG.get("leukemia_zip_files", []) if str(item).strip()]
    if zip_urls:
        os.environ["LEUKEMIA_ZIP_URLS"] = "\n".join(zip_urls)
    command = [
        sys.executable,
        "scripts/kaggle_download_detector_data.py",
        "--skip-txl",
        "--leukemia-url",
        str(CONFIG["leukemia_gdrive_folder_url"]),
        "--leukemia-output",
        str(CONFIG["leukemia_download_output"]),
        "--report-output",
        str(REPORT_DIR / "leukemiaattri_download.json"),
    ]
    for source in zip_files:
        command.extend(["--leukemia-zip-file", source])
    for source in zip_urls:
        command.extend(["--leukemia-zip-url", source])
    subprocess.run(command, check=True)
    CONFIG["leukemiaattri_root"] = CONFIG["leukemia_download_output"]
    return Path(CONFIG["leukemiaattri_root"])

def evaluate_logits(logits, labels, *, temperature=1.0, high_confidence_threshold=0.90):
    labels_t = torch.as_tensor(labels, dtype=torch.long)
    if len(labels_t) == 0:
        return {"status": "skipped", "reason": "empty labels"}
    scaled_logits = logits / float(temperature)
    probs_t = torch.softmax(scaled_logits, dim=-1)
    probs = probs_t.tolist()
    preds = probs_t.argmax(dim=1).tolist()
    labels_list = labels_t.tolist()
    report = classification_report(preds, labels_list, MLL23_CLASSES)
    cal = calibration_summary(probs, labels_list, bins=15)
    per_class = detailed_per_class_metrics(probs, labels_list, MLL23_CLASSES, high_confidence_threshold=high_confidence_threshold)
    high_conf_errors = sum(int(row["high_confidence_error_count"]) for row in per_class)
    return {
        "status": "ok",
        "accuracy": report["accuracy"],
        "macro_f1": report["macro_f1"],
        "balanced_accuracy": report["balanced_accuracy"],
        "top2_accuracy": top_k_accuracy(scaled_logits, labels_t, k=2),
        "ece": cal["expected_calibration_error"],
        "nll": cal["negative_log_likelihood"],
        "brier_score": cal["brier_score"],
        "high_confidence_error_count": high_conf_errors,
        "report": report,
        "calibration": cal,
        "confusion_matrix": confusion_matrix(preds, labels_list, len(MLL23_CLASSES)),
    }

def plot_confusion(matrix, title, output_path):
    if plt is None:
        return
    plt.figure(figsize=(10, 8))
    if sns is not None:
        sns.heatmap(matrix, cmap="Blues", cbar=True, xticklabels=MLL23_CLASSES, yticklabels=MLL23_CLASSES)
    else:
        plt.imshow(matrix, cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

def plot_reliability(calibration, title, output_path):
    if plt is None:
        return
    rows = calibration.get("reliability_bins", [])
    mids = [(row["low"] + row["high"]) / 2 for row in rows]
    acc = [row["accuracy"] for row in rows]
    conf = [row["confidence"] for row in rows]
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.plot(mids, acc, marker="o", label="accuracy")
    plt.plot(mids, conf, marker="o", label="confidence")
    plt.title(title)
    plt.xlabel("confidence bin")
    plt.ylabel("value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

    digest = hashlib.sha256()
    for part in json_safe(parts):
        digest.update(json.dumps(part, sort_keys=True).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

def require_paths(paths):
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("Required artifacts are missing. Attach them as Kaggle inputs or enable/download them in the Configuration cell. Missing: " + "; ".join(missing))

CONFIG["mll23_embeddings"] = resolve_path("mll23_embeddings", ["mll23_embeddings.pt"])
CONFIG["mll23_manifest"] = resolve_path("mll23_manifest", ["mll23_manifest.csv"])
CONFIG["mll23_splits"] = resolve_path("mll23_splits", ["mll23_splits.csv"])
CONFIG["classifier_v01_bundle"] = resolve_path("classifier_v01_bundle", ["mll23-dinobloom-b-mlp-v0.1"], must_be_dir=True)
CONFIG["leukemiaattri_root"] = maybe_download_leukemiaattri()

required_paths = [
    CONFIG["mll23_embeddings"], CONFIG["mll23_manifest"],
    CONFIG["mll23_splits"], CONFIG["classifier_v01_bundle"], CONFIG["leukemiaattri_root"],
]
require_paths(required_paths)
DINOBLOOM_WEIGHTS_PATH, DINOBLOOM_WEIGHTS_SHA256 = ensure_dinobloom_weights()
CONFIG["dinobloom_weights"] = DINOBLOOM_WEIGHTS_PATH

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None

def package_version(name):
    try:
        import importlib.metadata as metadata
        return metadata.version(name)
    except Exception:
        return None

protected_hashes_before = hash_files(CONFIG["classifier_v01_bundle"])
env_report = {
    "status": "ok",
    "utc_timestamp": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "packages": {name: package_version(name) for name in ["torch", "torchvision", "timm", "numpy", "pandas", "scikit-learn", "Pillow", "matplotlib", "seaborn"]},
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu": {
        "available": torch.cuda.is_available(),
        "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "memory_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
    },
    "git_commit": git_commit(),
    "configuration": CONFIG,
    "seeds": {"python": SEED, "numpy": SEED, "torch_cpu": SEED, "torch_cuda": SEED, "dataloader_worker_base": SEED},
    "source_dataset_paths": {key: str(CONFIG[key]) for key in ["mll23_manifest", "mll23_splits", "leukemiaattri_root"]},
    "protected_model_hashes": protected_hashes_before,
    "dinobloom_weights_hash": DINOBLOOM_WEIGHTS_SHA256,
    "dinobloom_download_url": CONFIG.get("dinobloom_download_url"),
    "dinobloom_expected_sha256": CONFIG.get("dinobloom_expected_sha256"),
    "deterministic_settings": {"cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic},
}
write_report(REPORT_DIR / "classifier_v0_2_environment.json", env_report)
print("Environment report:", REPORT_DIR / "classifier_v0_2_environment.json")







## 4. Protected `v0.1` integrity


In [ ]:
v01_bundle = Path(CONFIG["classifier_v01_bundle"])
v01_docs = load_bundle_documents(v01_bundle)
v01_inspection = inspect_bundle(v01_bundle)
v01_taxonomy = v01_docs["taxonomy"]
v01_class_names = list(v01_taxonomy.get("class_names", []))
verify_mll23_class_order(v01_class_names)
protected_hash_report = {
    "status": "ok",
    "bundle": str(v01_bundle),
    "inspection": v01_inspection,
    "class_names": v01_class_names,
    "hashes_before": protected_hashes_before,
}
write_report(REPORT_DIR / "classifier_v0_2_protected_v0_1_integrity.json", protected_hash_report)
print("Protected v0.1 taxonomy and hashes verified.")


## 5. Load and verify MLL23 artifacts


In [ ]:
def load_pt(path):
    return torch.load(path, map_location="cpu", weights_only=False)

mll23_embeddings_obj = load_pt(CONFIG["mll23_embeddings"])
mll23_manifest_df = pd.read_csv(CONFIG["mll23_manifest"])
mll23_splits_df = pd.read_csv(CONFIG["mll23_splits"])

def first_column(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None

def coerce_mll23_cache(obj):
    if isinstance(obj, dict):
        embeddings = obj.get("embeddings") or obj.get("features")
        labels = obj.get("labels") or obj.get("label_indices")
        crop_ids = obj.get("crop_ids") or obj.get("sample_ids") or obj.get("ids")
        splits = obj.get("splits")
    else:
        embeddings = getattr(obj, "embeddings", None)
        entries = getattr(obj, "entries", None)
        labels = [entry.label_index for entry in entries] if entries is not None else None
        crop_ids = [getattr(entry, "sample_id", str(i)) for i, entry in enumerate(entries)] if entries is not None else None
        splits = [getattr(entry, "split", None) for entry in entries] if entries is not None else None
    if embeddings is None or labels is None:
        raise RuntimeError("Unsupported MLL23 embedding artifact format. Expected embeddings plus labels/label_indices.")
    embeddings = torch.as_tensor(embeddings, dtype=torch.float32)
    labels = torch.as_tensor(labels, dtype=torch.long)
    if embeddings.ndim != 2 or embeddings.shape[1] != 768:
        raise RuntimeError(f"MLL23 embeddings must be [N, 768], got {tuple(embeddings.shape)}")
    if len(labels) != embeddings.shape[0]:
        raise RuntimeError("MLL23 label count does not match embedding rows")
    if crop_ids is None:
        id_col = first_column(mll23_manifest_df, ["crop_id", "sample_id", "image_id", "path", "relative_path"])
        if id_col and len(mll23_manifest_df) == embeddings.shape[0]:
            crop_ids = mll23_manifest_df[id_col].astype(str).tolist()
        else:
            crop_ids = [str(i) for i in range(embeddings.shape[0])]
    crop_ids = [str(value) for value in crop_ids]
    if len(set(crop_ids)) != len(crop_ids):
        raise RuntimeError("MLL23 embedding crop/sample IDs are not unique; refusing row-position-only joins")
    if splits is None:
        split_col = first_column(mll23_splits_df, ["split", "unified_split"])
        split_id_col = first_column(mll23_splits_df, ["crop_id", "sample_id", "image_id", "path", "relative_path"])
        if split_col is None:
            raise RuntimeError("mll23_splits.csv lacks a split column")
        if split_id_col is not None:
            split_by_id = dict(zip(mll23_splits_df[split_id_col].astype(str), mll23_splits_df[split_col].astype(str)))
            missing = [crop_id for crop_id in crop_ids if crop_id not in split_by_id]
            if missing:
                raise RuntimeError(f"MLL23 splits missing {len(missing)} embedding IDs; first={missing[:5]}")
            splits = [split_by_id[crop_id] for crop_id in crop_ids]
        elif len(mll23_splits_df) == len(crop_ids):
            splits = mll23_splits_df[split_col].astype(str).tolist()
        else:
            raise RuntimeError("Cannot align MLL23 splits to embeddings by ID or row count")
    return {"embeddings": embeddings, "labels": labels, "crop_ids": crop_ids, "splits": list(splits)}

mll23_cache = coerce_mll23_cache(mll23_embeddings_obj)
write_report(REPORT_DIR / "classifier_v0_2_mll23_artifact_inspection.json", {
    "status": "ok",
    "embedding_shape": list(mll23_cache["embeddings"].shape),
    "split_counts": Counter(mll23_cache["splits"]),
    "label_counts": Counter(int(x) for x in mll23_cache["labels"].tolist()),
    "id_alignment": "id_join_or_verified_row_count",
})
print("MLL23 embeddings loaded:", tuple(mll23_cache["embeddings"].shape))


## 6. Discover and audit LeukemiaAttri


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
root = Path(CONFIG["leukemiaattri_root"])
images = sorted(path for path in root.rglob("*") if path.suffix.lower() in IMAGE_EXTS and path.is_file())
json_annotations = sorted(path for path in root.rglob("*.json") if path.is_file())
yolo_labels = sorted(path for path in root.rglob("*.txt") if path.is_file())
image_by_stem = {path.stem: path for path in images}
image_records = []
read_errors = []
for path in tqdm(images, desc="Audit images"):
    try:
        with Image.open(path) as im:
            width, height = im.size
            mode = im.mode
        image_records.append({"image_path": str(path), "stem": path.stem, "width": width, "height": height, "mode": mode, "sha256": sha256_file(path)})
    except Exception as exc:
        read_errors.append({"image_path": str(path), "error": str(exc)})

def parse_coco(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    if not all(key in data for key in ["images", "annotations", "categories"]):
        return []
    categories = {int(row["id"]): str(row.get("name", row["id"])) for row in data["categories"]}
    images_by_id = {int(row["id"]): row for row in data["images"]}
    rows = []
    for ann in data["annotations"]:
        image_row = images_by_id.get(int(ann.get("image_id", -1)))
        if not image_row:
            continue
        file_name = Path(str(image_row.get("file_name", ""))).name
        image_path = next((img for img in images if img.name == file_name or img.stem == Path(file_name).stem), None)
        if image_path is None:
            continue
        x, y, w, h = [float(v) for v in ann.get("bbox", [0, 0, 0, 0])]
        rows.append({"source": "coco", "annotation_file": str(path), "image_path": str(image_path), "box": [x, y, x + w, y + h], "source_class": categories.get(int(ann.get("category_id", -1)), str(ann.get("category_id"))), "attributes": {k: v for k, v in ann.items() if k not in {"bbox", "category_id", "image_id"}}})
    return rows

def parse_yolo_label(path):
    image_path = image_by_stem.get(path.stem)
    if image_path is None:
        return []
    with Image.open(image_path) as im:
        width, height = im.size
    rows = []
    for line_no, line in enumerate(path.read_text(encoding="utf-8", errors="ignore").splitlines(), start=1):
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            class_name = parts[0]
            cx, cy, bw, bh = [float(v) for v in parts[1:5]]
        except ValueError:
            continue
        x1 = (cx - bw / 2) * width
        y1 = (cy - bh / 2) * height
        x2 = (cx + bw / 2) * width
        y2 = (cy + bh / 2) * height
        rows.append({"source": "yolo", "annotation_file": str(path), "line_number": line_no, "image_path": str(image_path), "box": [x1, y1, x2, y2], "source_class": class_name, "attributes": {"extra_fields": parts[5:]}})
    return rows

annotations = []
for path in tqdm(json_annotations, desc="Parse COCO JSON"):
    try:
        annotations.extend(parse_coco(path))
    except Exception as exc:
        read_errors.append({"annotation_file": str(path), "error": str(exc)})
for path in tqdm(yolo_labels, desc="Parse YOLO labels"):
    try:
        annotations.extend(parse_yolo_label(path))
    except Exception as exc:
        read_errors.append({"annotation_file": str(path), "error": str(exc)})

invalid_boxes = []
class_counts = Counter()
for row in annotations:
    class_counts[row["source_class"]] += 1
    x1, y1, x2, y2 = row["box"]
    if x2 <= x1 or y2 <= y1:
        invalid_boxes.append({**row, "reason": "non_positive_box"})
    try:
        with Image.open(row["image_path"]) as im:
            width, height = im.size
        if x2 < 0 or y2 < 0 or x1 > width or y1 > height:
            invalid_boxes.append({**row, "reason": "box_outside_image"})
    except Exception:
        pass

hash_counts = Counter(row["sha256"] for row in image_records)
duplicate_hashes = {h: c for h, c in hash_counts.items() if c > 1}
critical_errors = []
if read_errors:
    critical_errors.append("unreadable_inputs")
if not annotations:
    critical_errors.append("no_annotations_discovered")
if invalid_boxes:
    critical_errors.append("invalid_boxes")

audit = {
    "status": "fail" if critical_errors else "ok",
    "critical_errors": critical_errors,
    "image_count": len(images),
    "readable_image_count": len(image_records),
    "annotation_count": len(annotations),
    "json_annotation_files": [str(p) for p in json_annotations],
    "yolo_label_files": [str(p) for p in yolo_labels],
    "source_class_counts": class_counts,
    "duplicate_image_hashes": duplicate_hashes,
    "read_errors": read_errors[:100],
    "invalid_boxes": invalid_boxes[:100],
    "patient_slide_field_grouping": "not_verified_without_source_metadata",
    "license_provenance": "inspect source Kaggle dataset metadata manually and record before promotion",
    "annotation_completeness": "not_verified_without official documentation",
}
write_report(REPORT_DIR / "leukemiaattri_classifier_audit.json", audit)
(REPORT_DIR / "leukemiaattri_classifier_audit.md").write_text("# LeukemiaAttri classifier audit\n\n" + json.dumps(json_safe(audit), indent=2), encoding="utf-8")

gallery = REPORT_DIR / "leukemiaattri_classifier_overlay_gallery"
gallery.mkdir(parents=True, exist_ok=True)
for idx, row in enumerate(annotations[:24]):
    try:
        with Image.open(row["image_path"]).convert("RGB") as im:
            draw = ImageDraw.Draw(im)
            draw.rectangle(row["box"], outline="lime", width=3)
            draw.text((row["box"][0], max(0, row["box"][1]-12)), str(row["source_class"]), fill="lime")
            im.thumbnail((768, 768))
            im.save(gallery / f"sample_{idx:03d}.png")
    except Exception:
        pass
print("Audit status:", audit["status"], audit["critical_errors"])
if audit["status"] != "ok":
    raise RuntimeError("Critical LeukemiaAttri audit errors remain; refusing to train. See leukemiaattri_classifier_audit.json")


## 7. Taxonomy harmonization


In [ ]:
class_to_index = {name: idx for idx, name in enumerate(MLL23_CLASSES)}
source_classes = sorted(class_counts)
mapping_rows = [decide_source_mapping(name).to_dict() for name in source_classes]
mapping_report = {
    "status": "ok",
    "mll23_classes": MLL23_CLASSES,
    "mappings": mapping_rows,
    "provenance_review_required": [row for row in mapping_rows if row["requires_provenance_review"]],
    "unmapped_classes": [row for row in mapping_rows if row["mapping_type"].startswith("unmapped")],
    "partial_label_classes": [row for row in mapping_rows if row["mapping_type"] == "partial"],
    "outlier_exposure_enabled": bool(CONFIG["allow_outlier_exposure"]),
}
write_report(REPORT_DIR / "leukemiaattri_taxonomy_mapping.json", mapping_report)
pd.DataFrame(mapping_rows).to_csv(MANIFEST_DIR / "leukemiaattri_taxonomy_mapping.csv", index=False)
display(pd.DataFrame(mapping_rows))



## 8. Group-aware split construction


In [ ]:
annotation_rows = []
image_hash_by_path = {row["image_path"]: row["sha256"] for row in image_records}
for idx, ann in enumerate(annotations):
    decision = decide_source_mapping(ann["source_class"])
    image_path = ann["image_path"]
    source_hash = image_hash_by_path.get(image_path) or sha256_file(image_path)
    acquisition_group = Path(image_path).stem
    annotation_rows.append({
        "annotation_id": f"la_{idx:08d}",
        "source_image": image_path,
        "source_image_sha256": source_hash,
        "source_box": json.dumps([round(float(v), 4) for v in ann["box"]]),
        "source_class": ann["source_class"],
        "mapping_type": decision.mapping_type,
        "target_class": decision.target_class,
        "partial_target_classes": json.dumps(list(decision.partial_target_classes)),
        "patient_id": "unknown",
        "slide_id": "unknown",
        "field_id": Path(image_path).stem,
        "acquisition_group": acquisition_group,
        "microscope": "unknown",
        "camera": "unknown",
        "magnification": "unknown",
        "attributes": json.dumps(ann.get("attributes", {})),
    })
splits = group_aware_split(annotation_rows, group_key="acquisition_group", seed=SEED)
for row, split in zip(annotation_rows, splits):
    row["split"] = split
leakage = split_leakage_report(annotation_rows, "split", ["source_image_sha256", "source_image", "acquisition_group"])
write_report(REPORT_DIR / "classifier_v0_2_split_integrity.json", {"status": leakage["status"], "leakage": leakage, "split_counts": Counter(splits), "patient_level_separation": "not_verified"})
pd.DataFrame(annotation_rows).to_csv(MANIFEST_DIR / "leukemiaattri_source_annotations.csv", index=False)
if leakage["status"] != "ok":
    raise RuntimeError("Confirmed split leakage detected; refusing to train. See classifier_v0_2_split_integrity.json")
print("Split counts:", Counter(splits))


## 9. Ground-truth crop generation


In [ ]:
def render_square_crop(image_path, box, padding_ratio, output_path):
    with Image.open(image_path).convert("RGB") as image:
        geometry = square_crop_geometry(box, image.width, image.height, padding_ratio)
        crop = image.crop(geometry.crop_box)
        canvas = Image.new("RGB", (geometry.output_size, geometry.output_size), (0, 0, 0))
        canvas.paste(crop, geometry.paste_box)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        canvas.save(output_path)
        return geometry

selected_padding = float(CONFIG["selected_padding_fallback"])
crop_rows = []
for row in tqdm(annotation_rows, desc="Generate default crops"):
    crop_id = row["annotation_id"] + f"_pad{selected_padding:.2f}".replace(".", "p")
    crop_path = CROP_DIR / "selected_padding" / f"{crop_id}.png"
    box = json.loads(row["source_box"])
    geometry = render_square_crop(row["source_image"], box, selected_padding, crop_path)
    crop_rows.append({**row, "crop_id": crop_id, "padding_ratio": selected_padding, "crop_path": str(crop_path), "crop_geometry": json.dumps(geometry.to_dict())})
crop_df = pd.DataFrame(crop_rows)
crop_manifest = MANIFEST_DIR / "leukemiaattri_classifier_crops.csv"
crop_df.to_csv(crop_manifest, index=False)
training_manifest = MANIFEST_DIR / "classifier_v0_2_training_manifest.csv"
crop_df.to_csv(training_manifest, index=False)
print("Crops:", len(crop_df), crop_manifest)


## 10. Padding sweep


In [ ]:
def build_v01_head_for_eval():
    bundle = v01_docs["bundle"]
    head = build_head(str(bundle["head"]), len(MLL23_CLASSES))
    ckpt = torch.load(v01_bundle / str(bundle["head_file"]), map_location="cpu", weights_only=True)
    state = ckpt.get("model_state", ckpt) if isinstance(ckpt, dict) else ckpt
    head.load_state_dict(state)
    head.eval()
    return head.to(device)

def extract_embeddings_for_crop_paths(crop_paths, batch_size):
    model = load_dinobloom_b_backbone(CONFIG["dinobloom_weights"], device=device)
    model.eval()
    output = []
    for start in tqdm(range(0, len(crop_paths), batch_size), desc="Extract padding-sweep embeddings"):
        batch_paths = crop_paths[start:start + batch_size]
        tensors = [preprocess_crop(path)[0] for path in batch_paths]
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            output.append(extract_backbone_embedding(model, batch).detach().cpu())
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return torch.cat(output, dim=0) if output else torch.empty((0, 768), dtype=torch.float32)

padding_report_path = REPORT_DIR / "leukemiaattri_padding_sweep.json"
padding_fingerprint = artifact_fingerprint({
    "stage": "padding_sweep_v2",
    "padding_ratios": CONFIG["padding_ratios"],
    "annotations": stable_sample_hash([row["annotation_id"] + row["source_image"] + row["source_box"] for row in annotation_rows[:1000]]),
    "backbone_hash": DINOBLOOM_WEIGHTS_SHA256,
    "v01_hash": protected_hashes_before,
})
if padding_report_path.exists():
    existing_padding = json.loads(padding_report_path.read_text(encoding="utf-8"))
    if existing_padding.get("fingerprint") == padding_fingerprint and existing_padding.get("status") == "ok":
        padding_report = existing_padding
        selected_padding = float(padding_report["selected_padding"])
    else:
        padding_report_path.unlink()

if not padding_report_path.exists():
    padding_rows = []
    v01_head = build_v01_head_for_eval()
    val_rows = [row for row in annotation_rows if row["split"] == "validation" and row["mapping_type"] in {"exact", "partial"}]
    if not val_rows:
        raise RuntimeError("No compatible LeukemiaAttri validation rows for padding selection")
    for ratio in CONFIG["padding_ratios"]:
        ratio_dir = CROP_DIR / "padding_sweep" / f"pad_{ratio:.2f}".replace(".", "p")
        ratio_dir.mkdir(parents=True, exist_ok=True)
        crop_paths = []
        exact_indices = []
        exact_labels = []
        partial_indices = []
        for local_index, row in enumerate(tqdm(val_rows, desc=f"Render padding {ratio}")):
            crop_path = ratio_dir / f"{row['annotation_id']}.png"
            if not crop_path.exists():
                render_square_crop(row["source_image"], json.loads(row["source_box"]), float(ratio), crop_path)
            crop_paths.append(str(crop_path))
            if row["mapping_type"] == "exact" and row["target_class"] in class_to_index:
                exact_indices.append(local_index)
                exact_labels.append(class_to_index[row["target_class"]])
            elif row["mapping_type"] == "partial":
                partial_indices.append(local_index)
        embeddings = extract_embeddings_for_crop_paths(crop_paths, int(CONFIG["batch_size"]))
        with torch.no_grad():
            logits = v01_head(embeddings.to(device)).detach().cpu()
            probs = torch.softmax(logits, dim=1)
        exact_metrics = {"status": "skipped", "reason": "no exact-compatible validation rows"}
        if exact_indices:
            exact_logits = logits[torch.as_tensor(exact_indices)]
            exact_probs = probs[torch.as_tensor(exact_indices)]
            exact_metrics = evaluate_logits(exact_logits, exact_labels)
            exact_metrics["expected_class_probability"] = float(torch.tensor([exact_probs[i, label] for i, label in enumerate(exact_labels)]).mean())
        partial_metrics = {"status": "skipped", "reason": "no broad-neutrophil validation rows"}
        if partial_indices:
            partial_probs = probs[torch.as_tensor(partial_indices)]
            grouped = partial_probs[:, [class_to_index["neutrophil_band"], class_to_index["neutrophil_segmented"]]].sum(dim=1)
            partial_metrics = {"status": "ok", "count": len(partial_indices), "mean_grouped_probability": float(grouped.mean())}
        padding_rows.append({
            "padding_ratio": float(ratio),
            "validation_crop_count": len(val_rows),
            "exact_metrics": exact_metrics,
            "partial_neutrophil_metrics": partial_metrics,
            "selection_inputs": ["compatible_class_macro_f1", "expected_class_probability", "crop_completeness", "multi_cell_contamination_review"],
        })
    del v01_head
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    selected = max(
        padding_rows,
        key=lambda row: (
            row["exact_metrics"].get("report", {}).get("macro_f1", -1),
            row["exact_metrics"].get("expected_class_probability", -1),
            -abs(float(row["padding_ratio"]) - 0.25),
        ),
    )
    selected_padding = float(selected["padding_ratio"])
    padding_report = {"status": "ok", "fingerprint": padding_fingerprint, "selected_padding": selected_padding, "selection_reason": "validation macro-F1, expected-class probability, and crop-review prior", "candidates": padding_rows}
    write_report(padding_report_path, padding_report)

(REPORT_DIR / "leukemiaattri_padding_sweep.md").write_text("# LeukemiaAttri padding sweep\n\n" + json.dumps(json_safe(padding_report), indent=2), encoding="utf-8")
if plt is not None:
    rows = padding_report["candidates"]
    plt.figure(figsize=(7, 4))
    plt.plot([r["padding_ratio"] for r in rows], [r["exact_metrics"].get("report", {}).get("macro_f1", 0.0) for r in rows], marker="o", label="exact macro-F1")
    plt.plot([r["padding_ratio"] for r in rows], [r["exact_metrics"].get("expected_class_probability", 0.0) for r in rows], marker="o", label="expected-class probability")
    plt.axvline(selected_padding, color="black", linestyle="--", label="selected")
    plt.xlabel("padding ratio")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "leukemiaattri_padding_comparison.png", dpi=150)
    plt.savefig(PLOT_DIR / "padding_comparison.png", dpi=150)
print("Selected padding:", selected_padding)


## 11. DinoBloom initialization and parity


In [ ]:
backbone = load_dinobloom_b_backbone(CONFIG["dinobloom_weights"], device=device)
backbone.eval()
parity_rows = []
parity_status = "skipped"
parity_reason = "No MLL23 crop paths could be resolved from manifest. Training can continue only if embeddings artifact provenance is trusted."
manifest_id_col = first_column(mll23_manifest_df, ["crop_id", "sample_id", "image_id", "path", "relative_path"])
path_col = first_column(mll23_manifest_df, ["crop_path", "image_path", "path", "relative_path"])
if manifest_id_col and path_col:
    path_by_id = dict(zip(mll23_manifest_df[manifest_id_col].astype(str), mll23_manifest_df[path_col].astype(str)))
    checked = 0
    for idx, crop_id in enumerate(mll23_cache["crop_ids"]):
        if crop_id not in path_by_id:
            continue
        candidate_path = Path(path_by_id[crop_id])
        if not candidate_path.is_absolute():
            candidate_path = REPO_ROOT / candidate_path
        if not candidate_path.exists():
            continue
        with torch.no_grad():
            live = extract_backbone_embedding(backbone, preprocess_crop(candidate_path).to(device)).detach().cpu()[0]
        cached = mll23_cache["embeddings"][idx]
        max_abs_diff = float((live - cached).abs().max())
        parity_rows.append({"crop_id": crop_id, "crop_path": str(candidate_path), "max_abs_diff": max_abs_diff})
        checked += 1
        if checked >= 8:
            break
    if parity_rows:
        worst = max(row["max_abs_diff"] for row in parity_rows)
        parity_status = "ok" if worst <= 1e-4 else "fail"
        parity_reason = None if parity_status == "ok" else f"live-vs-cached embedding max_abs_diff {worst} exceeds tolerance"
if parity_status == "fail":
    raise RuntimeError(parity_reason)
parity_report = {"status": parity_status, "reason": parity_reason, "device": device, "checked_count": len(parity_rows), "checks": parity_rows, "preprocessing": PREPROCESSING, "backbone_hash": sha256_file(CONFIG["dinobloom_weights"])}
write_report(REPORT_DIR / "classifier_v0_2_dinobloom_parity.json", parity_report)
print("Backbone loaded on", device, "parity:", parity_status)


## 12. LeukemiaAttri embedding extraction


In [ ]:
embedding_path = EMBEDDING_DIR / "leukemiaattri_embeddings.pt"
if embedding_path.exists():
    leukemia_cache = torch.load(embedding_path, map_location="cpu", weights_only=False)
    print("Loaded existing embeddings", embedding_path)
else:
    embeddings = []
    kept_rows = []
    skipped = []
    backbone.eval()
    for start in tqdm(range(0, len(crop_rows), CONFIG["batch_size"]), desc="Extract LeukemiaAttri embeddings"):
        batch_rows = crop_rows[start:start + CONFIG["batch_size"]]
        tensors = []
        ok_rows = []
        for row in batch_rows:
            try:
                tensors.append(preprocess_crop(row["crop_path"])[0])
                ok_rows.append(row)
            except Exception as exc:
                skipped.append({"crop_id": row["crop_id"], "error": str(exc)})
        if not tensors:
            continue
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            emb = extract_backbone_embedding(backbone, batch).detach().cpu()
        embeddings.append(emb)
        kept_rows.extend(ok_rows)
    if not embeddings:
        raise RuntimeError("No LeukemiaAttri embeddings were extracted")
    embedding_tensor = torch.cat(embeddings, dim=0)
    if embedding_tensor.shape[1] != 768 or not torch.isfinite(embedding_tensor).all():
        raise RuntimeError("Invalid LeukemiaAttri embeddings")
    leukemia_cache = {
        "embeddings": embedding_tensor,
        "rows": kept_rows,
        "crop_ids": [row["crop_id"] for row in kept_rows],
        "splits": [row["split"] for row in kept_rows],
        "mapping_types": [row["mapping_type"] for row in kept_rows],
        "target_classes": [row["target_class"] for row in kept_rows],
        "partial_target_classes": [json.loads(row["partial_target_classes"]) for row in kept_rows],
        "preprocessing": PREPROCESSING,
        "backbone_hash": sha256_file(CONFIG["dinobloom_weights"]),
        "selected_padding": selected_padding,
        "skipped": skipped,
    }
    torch.save(leukemia_cache, embedding_path)
write_report(REPORT_DIR / "leukemiaattri_embeddings.json", {"status": "ok", "path": str(embedding_path), "row_count": len(leukemia_cache["crop_ids"]), "embedding_shape": list(leukemia_cache["embeddings"].shape), "split_counts": Counter(leukemia_cache["splits"]), "mapping_type_counts": Counter(leukemia_cache["mapping_types"]), "skipped_count": len(leukemia_cache.get("skipped", [])), "sample_hash": stable_sample_hash(leukemia_cache["crop_ids"][:100])})
del backbone
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("LeukemiaAttri embeddings:", tuple(leukemia_cache["embeddings"].shape))


## 13. Dataset balancing


In [ ]:
la_embeddings = leukemia_cache["embeddings"]
la_rows = leukemia_cache["rows"]

def row_target_index(row):
    return class_to_index[row["target_class"]] if row.get("target_class") in class_to_index else None

def mll23_indices(split):
    return [idx for idx, value in enumerate(mll23_cache["splits"]) if value == split]

def la_indices(split, mapping_types):
    return [idx for idx, row in enumerate(la_rows) if row["split"] == split and row["mapping_type"] in mapping_types]

balancing_report = {
    "mll23_split_counts": Counter(mll23_cache["splits"]),
    "leukemiaattri_split_counts": Counter(row["split"] for row in la_rows),
    "leukemiaattri_mapping_counts": Counter(row["mapping_type"] for row in la_rows),
    "lambda_mll23": CONFIG["lambda_mll23"],
    "lambda_leukemia_exact": CONFIG["lambda_leukemia_exact"],
    "lambda_leukemia_partial": CONFIG["lambda_leukemia_partial"],
}
write_report(REPORT_DIR / "classifier_v0_2_dataset_balancing.json", {"status": "ok", **balancing_report})
print(json.dumps(json_safe(balancing_report), indent=2))



## 14. Head definitions


In [ ]:
def load_v01_head():
    bundle = v01_docs["bundle"]
    head = build_head(str(bundle["head"]), len(MLL23_CLASSES))
    ckpt = torch.load(v01_bundle / str(bundle["head_file"]), map_location="cpu", weights_only=True)
    state = ckpt.get("model_state", ckpt) if isinstance(ckpt, dict) else ckpt
    head.load_state_dict(state)
    return head

def init_head(warm_start):
    if warm_start:
        return load_v01_head()
    return build_head(str(v01_docs["bundle"]["head"]), len(MLL23_CLASSES))

head_definition_report = {"status": "ok", "head": v01_docs["bundle"]["head"], "num_outputs": len(MLL23_CLASSES), "class_order": MLL23_CLASSES}
write_report(REPORT_DIR / "classifier_v0_2_head_definition.json", head_definition_report)


## 15. Candidate A training


In [ ]:
def candidate_data(include_partial):
    m_train = mll23_indices("train")
    exact_train = la_indices("train", {"exact"})
    partial_train = la_indices("train", {"partial"}) if include_partial else []
    return m_train, exact_train, partial_train

def validation_objective(model):
    model.eval()
    scores = {}
    with torch.no_grad():
        m_val = mll23_indices("validation")
        if m_val:
            logits = model(mll23_cache["embeddings"][torch.as_tensor(m_val)].to(device)).detach().cpu()
            labels = mll23_cache["labels"][m_val].tolist()
            scores["mll23"] = evaluate_logits(logits, labels)
        exact_val = la_indices("validation", {"exact"})
        if exact_val:
            logits = model(la_embeddings[torch.as_tensor(exact_val)].to(device)).detach().cpu()
            labels = [row_target_index(la_rows[i]) for i in exact_val]
            scores["leukemia_exact"] = evaluate_logits(logits, labels)
        partial_val = la_indices("validation", {"partial"})
        if partial_val:
            logits = model(la_embeddings[torch.as_tensor(partial_val)].to(device)).detach().cpu()
            probs = torch.softmax(logits, dim=1)
            grouped = probs[:, [class_to_index["neutrophil_band"], class_to_index["neutrophil_segmented"]]].sum(dim=1)
            scores["leukemia_partial"] = {"status": "ok", "mean_grouped_probability": float(grouped.mean()), "count": len(partial_val)}
    mll23_macro = scores.get("mll23", {}).get("macro_f1", -1.0)
    exact_macro = scores.get("leukemia_exact", {}).get("macro_f1", -1.0)
    partial_prob = scores.get("leukemia_partial", {}).get("mean_grouped_probability", 0.0)
    objective = (mll23_macro, exact_macro, partial_prob)
    return objective, scores

def shuffled_batches(indices, batch_size, generator):
    if not indices:
        return []
    order = torch.as_tensor(indices)[torch.randperm(len(indices), generator=generator)].tolist()
    return [order[start:start + batch_size] for start in range(0, len(order), batch_size)]

def train_candidate(name, *, include_partial, warm_start):
    checkpoint_path = CHECKPOINT_DIR / f"classifier_v0_2_{name}.pt"
    history_path = REPORT_DIR / f"classifier_v0_2_{name}_history.json"
    train_fingerprint = artifact_fingerprint({
        "experiment": name,
        "include_partial": include_partial,
        "warm_start": warm_start,
        "selected_padding": selected_padding,
        "backbone_hash": DINOBLOOM_WEIGHTS_SHA256,
        "v01_hash": protected_hashes_before,
        "mll23_ids": stable_sample_hash(mll23_cache["crop_ids"][:1000]),
        "leukemia_ids": stable_sample_hash([row["crop_id"] for row in la_rows[:1000]]),
        "config": {k: CONFIG[k] for k in ["batch_size", "epochs", "learning_rate", "weight_decay", "lambda_mll23", "lambda_leukemia_exact", "lambda_leukemia_partial"]},
    })
    if checkpoint_path.exists() and history_path.exists():
        existing = json.loads(history_path.read_text(encoding="utf-8"))
        if existing.get("fingerprint") == train_fingerprint:
            return existing
    model = init_head(warm_start).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    m_train, exact_train, partial_train = candidate_data(include_partial)
    if not (m_train or exact_train or partial_train):
        raise RuntimeError(f"No training rows for {name}")
    history = []
    best_objective = (-1.0, -1.0, -1.0)
    best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    best_scores = {}
    stale = 0
    component_batch = max(1, int(CONFIG["batch_size"]) // 3)
    for epoch in range(int(CONFIG["epochs"])):
        model.train()
        generator = torch.Generator().manual_seed(SEED + epoch)
        m_batches = shuffled_batches(m_train, component_batch, generator)
        exact_batches = shuffled_batches(exact_train, component_batch, generator)
        partial_batches = shuffled_batches(partial_train, component_batch, generator)
        step_count = max(len(m_batches), len(exact_batches), len(partial_batches), 1)
        epoch_losses = []
        for step in range(step_count):
            optimizer.zero_grad()
            losses = []
            if m_batches:
                batch_idx = m_batches[step % len(m_batches)]
                logits = model(mll23_cache["embeddings"][torch.as_tensor(batch_idx)].to(device))
                labels = mll23_cache["labels"][batch_idx].to(device)
                losses.append(float(CONFIG["lambda_mll23"]) * F.cross_entropy(logits, labels))
            if exact_batches:
                batch_idx = exact_batches[step % len(exact_batches)]
                logits = model(la_embeddings[torch.as_tensor(batch_idx)].to(device))
                labels = torch.tensor([row_target_index(la_rows[i]) for i in batch_idx], dtype=torch.long, device=device)
                losses.append(float(CONFIG["lambda_leukemia_exact"]) * F.cross_entropy(logits, labels))
            if partial_batches:
                batch_idx = partial_batches[step % len(partial_batches)]
                logits = model(la_embeddings[torch.as_tensor(batch_idx)].to(device))
                target_indices = [class_to_index["neutrophil_band"], class_to_index["neutrophil_segmented"]]
                losses.append(float(CONFIG["lambda_leukemia_partial"]) * partial_label_loss(logits, target_indices, torch=torch))
            loss = sum(losses)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(CONFIG["gradient_clip_norm"]))
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        objective, validation_scores = validation_objective(model)
        improved = objective > best_objective
        history.append({"epoch": epoch, "train_loss": sum(epoch_losses) / len(epoch_losses), "validation_objective": list(objective), "validation_scores": validation_scores})
        if improved:
            best_objective = objective
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_scores = validation_scores
            stale = 0
        else:
            stale += 1
            if stale >= int(CONFIG["patience"]):
                break
    model.load_state_dict(best_state)
    checkpoint = {"model_state": best_state, "head": v01_docs["bundle"]["head"], "class_names": MLL23_CLASSES, "experiment": name, "include_partial": include_partial, "warm_start": warm_start, "fingerprint": train_fingerprint}
    torch.save(checkpoint, checkpoint_path)
    report = {"status": "ok", "fingerprint": train_fingerprint, "experiment": name, "checkpoint": str(checkpoint_path), "history": history, "best_validation_objective": list(best_objective), "best_validation_scores": best_scores, "train_sizes": {"mll23": len(m_train), "leukemia_exact": len(exact_train), "leukemia_partial": len(partial_train)}}
    write_report(history_path, report)
    return report

experiments = []
if CONFIG["run_all_four_candidates"]:
    for name, include_partial, warm in [("A_warm", False, True), ("A_clean", False, False)]:
        experiments.append(train_candidate(name, include_partial=include_partial, warm_start=warm))
else:
    experiments.append({"status": "skipped", "experiment": "A", "reason": "run_all_four_candidates disabled; prioritizing B experiments"})
print([row["experiment"] for row in experiments])


## 16. Candidate B partial-label training


In [ ]:
for name, include_partial, warm in [("B_warm", True, True), ("B_clean", True, False)]:
    if CONFIG["run_all_four_candidates"] or name in {"B_warm", "B_clean"}:
        experiments.append(train_candidate(name, include_partial=include_partial, warm_start=warm))
write_report(REPORT_DIR / "classifier_v0_2_training_comparison.json", {"status": "ok", "experiments": experiments})
if plt is not None:
    plt.figure(figsize=(8, 5))
    for exp in experiments:
        if exp.get("status") == "ok":
            xs = [row["epoch"] for row in exp["history"]]
            ys = [row["loss"] for row in exp["history"]]
            plt.plot(xs, ys, label=exp["experiment"])
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "training_curves.png", dpi=150)
print("Training reports saved")


## 17. Validation comparison


In [ ]:
def load_candidate_head(experiment):
    ckpt = torch.load(CHECKPOINT_DIR / f"classifier_v0_2_{experiment}.pt", map_location="cpu", weights_only=True)
    model = build_head(str(v01_docs["bundle"]["head"]), len(MLL23_CLASSES))
    model.load_state_dict(ckpt["model_state"])
    return model

def score_exact_leukemia(model, split, *, temperature=1.0):
    idxs = la_indices(split, {"exact"})
    if not idxs:
        return {"status": "skipped", "reason": f"no exact LeukemiaAttri {split} rows"}
    labels = [row_target_index(la_rows[i]) for i in idxs]
    with torch.no_grad():
        logits = model(la_embeddings[torch.as_tensor(idxs)].to(next(model.parameters()).device)).detach().cpu()
    return evaluate_logits(logits, labels, temperature=temperature)

def score_mll23(model, split, *, temperature=1.0):
    idxs = mll23_indices(split)
    if not idxs:
        return {"status": "skipped", "reason": f"no MLL23 {split} rows"}
    labels = mll23_cache["labels"][idxs].tolist()
    with torch.no_grad():
        logits = model(mll23_cache["embeddings"][torch.as_tensor(idxs)].to(next(model.parameters()).device)).detach().cpu()
    return evaluate_logits(logits, labels, temperature=temperature)

def grouped_neutrophil_score(model, split, *, temperature=1.0):
    idxs = la_indices(split, {"partial"})
    if not idxs:
        return {"status": "skipped", "reason": f"no broad neutrophil {split} rows"}
    with torch.no_grad():
        logits = model(la_embeddings[torch.as_tensor(idxs)].to(next(model.parameters()).device)).detach().cpu()
        probs = torch.softmax(logits / float(temperature), dim=1)
        grouped = probs[:, [class_to_index["neutrophil_band"], class_to_index["neutrophil_segmented"]]].sum(dim=1)
    return {"status": "ok", "count": len(idxs), "mean_grouped_probability": float(grouped.mean()), "median_grouped_probability": float(grouped.median())}

v01_baseline_head = load_v01_head().to(device)
baseline_validation = {
    "mll23_validation": score_mll23(v01_baseline_head, "validation"),
    "leukemia_exact_validation": score_exact_leukemia(v01_baseline_head, "validation"),
    "leukemia_partial_validation": grouped_neutrophil_score(v01_baseline_head, "validation"),
}
validation_rows = []
for exp in [e["experiment"] for e in experiments if e.get("status") == "ok"]:
    model = load_candidate_head(exp).to(device)
    validation_rows.append({
        "experiment": exp,
        "mll23_validation": score_mll23(model, "validation"),
        "leukemia_exact_validation": score_exact_leukemia(model, "validation"),
        "leukemia_partial_validation": grouped_neutrophil_score(model, "validation"),
    })
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
write_report(REPORT_DIR / "classifier_v0_2_validation_comparison.json", {"status": "ok", "baseline_v0_1": baseline_validation, "candidates": validation_rows})

def selection_key(row):
    mll23 = row["mll23_validation"]
    ext = row["leukemia_exact_validation"]
    partial = row["leukemia_partial_validation"]
    mll23_macro = mll23.get("macro_f1", -1) if mll23.get("status") == "ok" else -1
    ext_macro = ext.get("macro_f1", -1) if ext.get("status") == "ok" else -1
    partial_prob = partial.get("mean_grouped_probability", 0) if partial.get("status") == "ok" else 0
    ece = mll23.get("ece", 1) if mll23.get("status") == "ok" else 1
    return (mll23_macro, ext_macro, partial_prob, -ece)

ok_rows = [row for row in validation_rows if row["mll23_validation"].get("status") == "ok"]
if not ok_rows:
    raise RuntimeError("No valid candidate selected")
selected_row = max(ok_rows, key=selection_key)
selected_experiment = selected_row["experiment"]
shutil.copy2(CHECKPOINT_DIR / f"classifier_v0_2_{selected_experiment}.pt", CHECKPOINT_DIR / "classifier_v0_2_selected.pt")
print("Selected experiment:", selected_experiment)


## 18. Temperature calibration


In [ ]:
selected_head = load_candidate_head(selected_experiment).to(device)
val_idxs = mll23_indices("validation")
with torch.no_grad():
    val_logits = selected_head(mll23_cache["embeddings"][torch.as_tensor(val_idxs)].to(device)).detach().cpu()
val_labels = mll23_cache["labels"][val_idxs]
temperature = fit_temperature(val_logits, val_labels)
pre_probs = torch.softmax(val_logits, dim=1).tolist()
post_probs = torch.softmax(val_logits / temperature, dim=1).tolist()
policy = UncertaintyPolicy(**v01_docs["thresholds"])

def policy_metrics(probabilities, labels):
    if not labels:
        return {"status": "skipped", "reason": "empty labels"}
    decisions = [decide_prediction(row, MLL23_CLASSES, policy) for row in probabilities]
    top = [max(range(len(row)), key=row.__getitem__) for row in probabilities]
    accepted = [idx for idx, decision in enumerate(decisions) if decision.status == "accepted"]
    review = [idx for idx, decision in enumerate(decisions) if decision.status == "review"]
    unknown = [idx for idx, decision in enumerate(decisions) if decision.status == "unknown"]
    incorrect_accepted = sum(1 for idx in accepted if top[idx] != labels[idx])
    return {
        "status": "ok",
        "count": len(labels),
        "accepted_rate": len(accepted) / len(labels),
        "review_rate": len(review) / len(labels),
        "unknown_rate": len(unknown) / len(labels),
        "accepted_accuracy": (sum(1 for idx in accepted if top[idx] == labels[idx]) / len(accepted)) if accepted else None,
        "incorrect_accepted_count": incorrect_accepted,
        "threshold_source": "v0.1 policy evaluated on validation data",
    }

with torch.no_grad():
    baseline_val_logits = v01_baseline_head(mll23_cache["embeddings"][torch.as_tensor(val_idxs)].to(device)).detach().cpu()
baseline_val_probs = torch.softmax(baseline_val_logits, dim=1).tolist()
policy_validation = {
    "baseline_v0_1_mll23_validation": policy_metrics(baseline_val_probs, val_labels.tolist()),
    "candidate_v0_2_mll23_validation": policy_metrics(post_probs, val_labels.tolist()),
}
calibration_report = {
    "status": "ok",
    "selected_experiment": selected_experiment,
    "shared_temperature": temperature,
    "mll23_validation_ece_before": expected_calibration_error(pre_probs, val_labels.tolist(), bins=15),
    "mll23_validation_ece_after": expected_calibration_error(post_probs, val_labels.tolist(), bins=15),
    "decision_policy_validation": policy_validation,
    "threshold_decision": "reuse_v0.1_thresholds_only_if_acceptance_gates_pass_policy_validation",
}
write_report(REPORT_DIR / "classifier_v0_2_calibration.json", calibration_report)
print(calibration_report)


## 19. MLL23 test evaluation


In [ ]:
v01_baseline_head = load_v01_head().to(device)
baseline_mll23_test = score_mll23(v01_baseline_head, "test", temperature=1.0)
mll23_test = score_mll23(selected_head, "test", temperature=temperature)
write_report(REPORT_DIR / "classifier_v0_2_mll23_test.json", {
    "status": mll23_test.get("status", "ok"),
    "selected_experiment": selected_experiment,
    "baseline_v0_1": baseline_mll23_test,
    "candidate_v0_2": mll23_test,
    "baseline_reference": {"accuracy": 0.9440, "macro_f1": 0.8275, "balanced_accuracy": 0.8217, "top2_accuracy": 0.9881, "ece": 0.004605},
})
if mll23_test.get("status") == "ok":
    plot_confusion(mll23_test["confusion_matrix"], "MLL23 test confusion matrix", PLOT_DIR / "mll23_confusion_matrix.png")
    plot_reliability(mll23_test["calibration"], "MLL23 test reliability", PLOT_DIR / "mll23_reliability.png")
print(mll23_test)


## 20. LeukemiaAttri compatible evaluation


In [ ]:
baseline_leukemia_test = score_exact_leukemia(v01_baseline_head, "test", temperature=1.0)
leukemia_test = score_exact_leukemia(selected_head, "test", temperature=temperature)
baseline_partial_test = grouped_neutrophil_score(v01_baseline_head, "test", temperature=1.0)
partial_report = grouped_neutrophil_score(selected_head, "test", temperature=temperature)
write_report(REPORT_DIR / "classifier_v0_2_leukemiaattri_test.json", {
    "status": "ok",
    "selected_experiment": selected_experiment,
    "baseline_v0_1": {"exact_compatible": baseline_leukemia_test, "broad_neutrophil": baseline_partial_test},
    "candidate_v0_2": {"exact_compatible": leukemia_test, "broad_neutrophil": partial_report},
})
if leukemia_test.get("status") == "ok":
    plot_confusion(leukemia_test["confusion_matrix"], "LeukemiaAttri exact-compatible confusion matrix", PLOT_DIR / "leukemiaattri_confusion_matrix.png")
    plot_reliability(leukemia_test["calibration"], "LeukemiaAttri reliability", PLOT_DIR / "leukemiaattri_reliability.png")
print(leukemia_test, partial_report)


## 21. OOD/unmapped evaluation


In [ ]:
ood_idxs = la_indices("test", {"unmapped_ood", "unmapped_unknown"})
ood_report = {"status": "skipped", "reason": "no OOD/unmapped test rows"}
if ood_idxs:
    with torch.no_grad():
        logits = selected_head(la_embeddings[torch.as_tensor(ood_idxs)].to(device)).detach().cpu()
        probs = torch.softmax(logits / temperature, dim=1)
        max_probs, top = probs.max(dim=1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=1)
    ood_report = {"status": "ok", "row_count": len(ood_idxs), "mean_max_softmax": float(max_probs.mean()), "mean_entropy": float(entropy.mean()), "top_predicted_mll23_classes": Counter(MLL23_CLASSES[int(i)] for i in top.tolist())}
write_report(REPORT_DIR / "classifier_v0_2_ood.json", ood_report)
if plt is not None:
    plt.figure(figsize=(5, 3))
    if ood_idxs:
        plt.hist(max_probs.numpy(), bins=20)
    plt.title("OOD max softmax distribution")
    plt.tight_layout(); plt.savefig(PLOT_DIR / "ood_confidence_distribution.png", dpi=150)
print(ood_report)


## 22. Smoke-test evaluation


In [ ]:
smoke_report = {"status": "skipped", "reason": "No configured smoke-test crop directory in this notebook configuration; do not interpret as pass."}
write_report(REPORT_DIR / "classifier_v0_2_smoke_test.json", smoke_report)
print(smoke_report)


## 23. Acceptance decision


In [ ]:
def metric_delta(candidate, baseline, key, *, missing_is_failure=True):
    if candidate.get("status") != "ok" or baseline.get("status") != "ok":
        if missing_is_failure:
            return None
        return 0.0
    return float(candidate.get(key, 0.0)) - float(baseline.get(key, 0.0))

mll23_accuracy_delta = metric_delta(mll23_test, baseline_mll23_test, "accuracy")
mll23_macro_delta = metric_delta(mll23_test, baseline_mll23_test, "macro_f1")
external_macro_delta = metric_delta(leukemia_test, baseline_leukemia_test, "macro_f1")
high_conf_delta = None
if mll23_test.get("status") == "ok" and baseline_mll23_test.get("status") == "ok":
    high_conf_delta = int(mll23_test.get("high_confidence_error_count", 0)) - int(baseline_mll23_test.get("high_confidence_error_count", 0))
ece_delta = None
if mll23_test.get("status") == "ok" and baseline_mll23_test.get("status") == "ok":
    ece_delta = float(mll23_test.get("ece", 0.0)) - float(baseline_mll23_test.get("ece", 0.0))
missing_gate_inputs = [
    name for name, value in {
        "mll23_accuracy_delta": mll23_accuracy_delta,
        "mll23_macro_delta": mll23_macro_delta,
        "external_macro_delta": external_macro_delta,
        "high_conf_delta": high_conf_delta,
        "ece_delta": ece_delta,
    }.items() if value is None
]
metrics_for_acceptance = {
    "protected_hashes_changed": False,
    "split_leakage_detected": leakage["status"] != "ok",
    "all_18_output_classes_required": True,
    "mll23_accuracy_drop": max(0.0, -(mll23_accuracy_delta or 0.0)),
    "mll23_macro_f1_drop": max(0.0, -(mll23_macro_delta or 0.0)),
    "external_exact_macro_f1_improvement": external_macro_delta if external_macro_delta is not None else -1.0,
    "high_confidence_error_increased": high_conf_delta is None or high_conf_delta > 0,
    "incorrect_accepted_increased": (policy_validation["candidate_v0_2_mll23_validation"].get("incorrect_accepted_count") is None or policy_validation["baseline_v0_1_mll23_validation"].get("incorrect_accepted_count") is None or policy_validation["candidate_v0_2_mll23_validation"].get("incorrect_accepted_count", 0) > policy_validation["baseline_v0_1_mll23_validation"].get("incorrect_accepted_count", 0)),
    "calibration_materially_worse": ece_delta is None or ece_delta > 0.01,
    "missing_gate_inputs": missing_gate_inputs,
}
acceptance = acceptance_status(metrics_for_acceptance, CONFIG["acceptance_gates"])
if missing_gate_inputs and acceptance["status"] == "accepted_candidate":
    acceptance = {"status": "rejected_integrity", "failures": ["missing_gate_inputs", *missing_gate_inputs]}
write_report(REPORT_DIR / "classifier_v0_2_acceptance.json", {"status": acceptance["status"], "selected_experiment": selected_experiment, "metrics": metrics_for_acceptance, "gates": CONFIG["acceptance_gates"], "failures": acceptance["failures"]})
if plt is not None:
    labels = ["MLL23 acc", "MLL23 macro-F1", "External macro-F1", "MLL23 ECE"]
    values = [mll23_accuracy_delta or 0.0, mll23_macro_delta or 0.0, external_macro_delta or 0.0, -(ece_delta or 0.0)]
    plt.figure(figsize=(8, 4))
    plt.bar(labels, values)
    plt.axhline(0, color="black", linewidth=1)
    plt.xticks(rotation=20)
    plt.title("Candidate v0.2 metric deltas vs v0.1")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "candidate_metric_comparison.png", dpi=150)
    plt.close()
print("Final acceptance status:", acceptance)



## 24. Packaging and checksums


In [ ]:
protected_hashes_after = hash_files(CONFIG["classifier_v01_bundle"])
assert_hashes_unchanged(protected_hashes_before, protected_hashes_after)

if acceptance["status"] == "accepted_candidate":
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINT_DIR / "classifier_v0_2_selected.pt", MODEL_DIR / "head.pt")
    write_report(MODEL_DIR / "taxonomy.json", {"class_names": MLL23_CLASSES})
    write_report(MODEL_DIR / "preprocessing.json", PREPROCESSING)
    write_report(MODEL_DIR / "thresholds.json", v01_docs["thresholds"])
    write_report(MODEL_DIR / "calibration.json", {"temperature": temperature})
    write_report(MODEL_DIR / "metrics.json", {"acceptance": acceptance, "mll23_test": mll23_test, "leukemiaattri_test": leukemia_test})
    write_report(MODEL_DIR / "dataset_mapping.json", mapping_report)
    write_report(MODEL_DIR / "bundle.json", {"name": "mll23-leukemiaattri-dinobloom-b-mlp-v0.2", "research_only": True, "head": v01_docs["bundle"]["head"], "head_file": "head.pt"})
    (MODEL_DIR / "model_card.md").write_text("# mll23-leukemiaattri-dinobloom-b-mlp-v0.2\n\nResearch-only experimental candidate. Not clinically validated. Documents taxonomy harmonization, partial-label neutrophil loss, excluded unmapped classes, crop padding, split limitations, detector independence, calibration, and acceptance evidence in bundled reports.\n", encoding="utf-8")
    sums = {str(p.relative_to(MODEL_DIR)): sha256_file(p) for p in sorted(MODEL_DIR.rglob("*")) if p.is_file() and p.name != "sha256sums.txt"}
    (MODEL_DIR / "sha256sums.txt").write_text("\n".join(f"{v}  {k}" for k, v in sums.items()) + "\n", encoding="utf-8")
else:
    rejected_dir = OUTPUT_ROOT / "rejected_experimental_evidence"
    rejected_dir.mkdir(parents=True, exist_ok=True)
    write_report(rejected_dir / "README.json", {"status": acceptance["status"], "reason": "Acceptance gates did not pass; no production-like v0.2 bundle created."})

archive_path = Path("/kaggle/working/classifier_v0_2_outputs.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for base in [REPORT_DIR, PLOT_DIR, CHECKPOINT_DIR, MANIFEST_DIR, EMBEDDING_DIR, OUTPUT_ROOT / "rejected_experimental_evidence"]:
        if base.exists():
            for file in base.rglob("*"):
                if file.is_file() and file.suffix.lower() not in {".png", ".json", ".csv", ".pt", ".md", ".txt"}:
                    continue
                if file.is_file():
                    zf.write(file, file.relative_to(OUTPUT_ROOT))
    if acceptance["status"] == "accepted_candidate" and MODEL_DIR.exists():
        for file in MODEL_DIR.rglob("*"):
            if file.is_file():
                zf.write(file, file.relative_to(OUTPUT_ROOT))
archive_hash = sha256_file(archive_path)
summary = {"archive_path": str(archive_path), "archive_size_bytes": archive_path.stat().st_size, "archive_sha256": archive_hash, "final_status": acceptance["status"], "selected_experiment": selected_experiment, "limitations": ["LeukemiaAttri patient/slide grouping may be unverifiable from available metadata", "Do not claim improvement unless acceptance is accepted_candidate", "Unmapped classes are OOD evidence, not compatible accuracy"]}
write_report(REPORT_DIR / "classifier_v0_2_notebook_execution_summary.json", summary)
print(json.dumps(json_safe(summary), indent=2))


## 25. Final summary


In [ ]:
final_summary_path = REPORT_DIR / "classifier_v0_2_notebook_execution_summary.json"
print(final_summary_path.read_text(encoding="utf-8"))
print("Download archive:", "/kaggle/working/classifier_v0_2_outputs.zip")
